# 05 — LLM as Judge

## Why this notebook exists

In **notebook 04** we built a regression gate using only deterministic graders — exact match, `contains`, structured validation, golden outputs. Those graders are fast, free, and fully reproducible. But they break down the moment the task produces *open-ended text*: two summaries can be equally faithful and concise while sharing almost no words. An `exact_match` grader marks one correct and one wrong based purely on whether the string matches a reference. A `contains` grader tells us a keyword appears, not whether the output is actually good.

LLM-as-judge plugs that gap: we ask a language model to score an output against an explicit rubric, returning a structured score and a rationale we can read. This notebook introduces the technique, shows you how to wire it into the harness from notebook 03 as just another grader, and — critically — shows you where it goes wrong and how to check whether your judge can be trusted.

This is the **first notebook in the series that requires an OpenAI API key.** An early guard cell will stop and print instructions if the key is missing.

## What you'll learn

- Why deterministic graders mis-score open-ended outputs, motivating the need for a judge.
- How to set `OPENAI_API_KEY` and what the guard cell does when it is absent.
- How to define `make_llm_judge(rubric, client)` — a factory that returns a grader with the same `(example, output) -> Score` signature as every other grader in this series.
- How to write a concrete rubric, run the judge on good and bad outputs, and read the `Score` with its `rationale` field.
- How to plug `make_llm_judge` into `run_eval` alongside deterministic graders.
- How to define `judge_pairwise(client, prompt, output_a, output_b)` and why relative judgments are often more reliable than absolute scores.
- The three main traps: **position bias**, **verbosity bias**, and **self-preference / non-determinism** — with a concrete demonstration of position bias.
- How to validate the judge against a small human-labeled set and compute an agreement rate, so the judge is not just another untested component.